# Week 9: Building the panel

| | |
|---|---|
| **Due** | start of the Week 10 meeting |
| **Prerequisites** | Week 1, Week 6, Week 7, Week 8 |
| **Consumes** | UniProt REST for proteome `UP000000625`; AlphaFold DB models; your `week08_pocket_results.csv` |
| **Produces** | `week09_panel.csv`, `week09_drop_log.csv`, `week09_panel_pockets.csv` |

## Learning objectives

1. Retrieve a complete reviewed proteome from the UniProt REST application programming interface (API) using the streaming endpoint, and detect the silent truncation that the paginated endpoint produces instead.
2. Join UniProt accessions to AlphaFold models and to Protein Data Bank (PDB) entries, choosing inner versus left join per join and defending each choice.
3. Log every dropped record against a closed vocabulary of reason codes, and assert that input equals survivors plus drops.
4. Name the systematic bias the losses introduce, including at least three protein classes under-represented in the frozen panel
5. Run fpocket across the whole panel as a chunked, resumable batch job, report runtime, and produce `week09_panel_pockets.csv`.


**After this week the panel does not change.** 

In [ ]:
#@title Setup — run this first
%pip install -q "biopython>=1.85" "pandas>=2.0" "matplotlib>=3.7" "requests>=2.31"
print("setup complete")

In [ ]:
import pathlib, requests
FIXTURES = pathlib.Path("fixtures")
COURSE_DATA_URL = None  

def course_file(name: str) -> pathlib.Path:
    local = FIXTURES / name
    if local.exists():
        return local
    if COURSE_DATA_URL:
        FIXTURES.mkdir(exist_ok=True)
        r = requests.get(f"{COURSE_DATA_URL}/{name}", timeout=30); r.raise_for_status()
        local.write_bytes(r.content); return local
    raise FileNotFoundError(f"{name} not found. Put fixtures/ beside this notebook.")

print("fixtures:", FIXTURES.resolve())

In [ ]:
# ---- Which external tools do you have? -----------------------------------
# This cell installs nothing on your own computer. It checks what is present and,
# if something is missing, prints the exact command for YOUR platform.
# On Colab (a disposable virtual machine) it will offer to build tools for you.
import shutil, platform, sys, subprocess, textwrap

IS_COLAB = "google.colab" in sys.modules
OSNAME   = platform.system()                 # 'Darwin' = macOS, 'Windows', 'Linux'
IS_MAC   = OSNAME == "Darwin"
IS_WIN   = OSNAME == "Windows"
ARM_MAC  = IS_MAC and platform.machine() == "arm64"

def have(cmd):
    """True if an executable is on PATH."""
    return shutil.which(cmd) is not None

_HINT = {
    "fpocket": {
        "Darwin":  "conda install -c conda-forge fpocket",
        "Linux":   "conda install -c conda-forge fpocket",
        "Windows": ("No native Windows build of fpocket exists. Two options:\n"
                    "       (a) run this week's notebook on Google Colab, or\n"
                    "       (b) install WSL (Windows Subsystem for Linux) and work inside it."),
    },
    "obabel": {
        "Darwin":  "conda install -c conda-forge openbabel",
        "Linux":   "conda install -c conda-forge openbabel",
        "Windows": "conda install -c conda-forge openbabel   (a Windows build does exist)",
    },
    "foldseek": {
        "Darwin":  ("conda install -c bioconda foldseek\n"
                    "       On Apple Silicon the conda build may not solve; if so, download the\n"
                    "       official universal macOS binary from the Foldseek releases page."),
        "Linux":   "conda install -c bioconda foldseek",
        "Windows": ("No native Windows build. Use the Foldseek web server at\n"
                    "       https://search.foldseek.com/search"),
    },
    "vina": {
        "Darwin":  "pip install vina",
        "Linux":   "pip install vina",
        "Windows": "pip install vina   (if no wheel exists for your Python, use Colab for this week)",
    },
}

def check_tools(tools, colab_installer=None):
    """Report which of `tools` are available. Returns dict name -> bool.

    On Colab, optionally run `colab_installer()` to build what is missing.
    Locally, print per-platform instructions and return honestly.
    """
    print(f"platform: {OSNAME}{' (Apple Silicon)' if ARM_MAC else ''}"
          f"{' · Google Colab' if IS_COLAB else ' · local Jupyter'}")
    print()
    status = {t: have(t) for t in tools}
    for t, ok in status.items():
        print(f"  {'FOUND  ' if ok else 'MISSING'}  {t}")
    missing = [t for t, ok in status.items() if not ok]
    if not missing:
        print("\nAll required tools present.")
        return status

    if IS_COLAB and colab_installer is not None:
        print("\nOn Colab — attempting to install the missing tools "
              "(this VM is disposable, so this is safe).")
        try:
            colab_installer()
        except Exception as e:
            print("  install failed:", type(e).__name__, str(e)[:200])
        status = {t: have(t) for t in tools}
        missing = [t for t, ok in status.items() if not ok]
        if not missing:
            print("  all tools now present.")
            return status

    if missing:
        print("\n" + "=" * 68)
        print("MISSING TOOLS — nothing was installed on your machine.")
        print("Run the command(s) below in a terminal with your course environment")
        print("activated (conda activate compbiotech), then restart this kernel.")
        print("=" * 68)
        for t in missing:
            hint = _HINT.get(t, {}).get(OSNAME, "See the course setup guide.")
            print(f"\n  {t}:\n       {hint}")
        print()
    return status


In [ ]:
#@title Configuration — the ratified filter specification, in one place
import pandas as pd, numpy as np, pathlib, json, time, datetime, os, re, subprocess, shutil

# --- the five assigned Phase A drug targets. These are force-included. ----------------
ASSIGNED_TARGETS = {
    "P0ABQ4": "folA  dihydrofolate reductase        (trimethoprim)",
    "P0AC13": "folP  dihydropteroate synthase       (sulfamethoxazole)",
    "P0AEK4": "fabI  enoyl-ACP reductase            (triclosan)",
    "P0AES6": "gyrB  DNA gyrase subunit B           (novobiocin)",
    "P0AES4": "gyrA  DNA gyrase subunit A           (ciprofloxacin)",
}

FILTER_SPEC = {
    "proteome":        "UP000000625",   # E. coli K-12
    "reviewed_only":   True,            # Swiss-Prot only
    "min_length":      50,              # shorter entries are dropped
    "max_length":      None,            # NO UPPER LIMIT 
    "require_af_model": True,           # no model, no receptor, no docking
    "drop_fragments":  True,
    "drop_nonstandard_residues": True,
    "deduplicate_by_sequence":  True,
    "target_panel_size": 400,           # 300-500; fallback 150 if timing demands
    "seed":            20250909,        # RECORDED
}

# A naive length ceiling would silently delete gyrA (875 aa) and gyrB (804 aa) and make
# Phase A impossible. The ceiling is not "large", it is ABSENT.
assert FILTER_SPEC["max_length"] is None, "There is no upper length limit. Do not add one."

# --- the closed vocabulary of drop reasons. Nothing outside this list. ----------------
DROP_REASONS = (
    "NO_AF_MODEL",            # AlphaFold DB has no model for this accession
    "LENGTH_OUT_OF_RANGE",    # below min_length (no upper bound exists, so only this way)
    "FRAGMENT",               # UniProt marks the sequence as a fragment
    "NONSTANDARD_RESIDUE",    # U, O, X, B or Z in the sequence
    "OBSOLETE_OR_DEMERGED",   # accession no longer resolves in the current release
    "DUPLICATE_SEQUENCE",     # identical sequence to an entry already kept
    "NOT_SAMPLED",            # survived every filter; the seeded subsample did not pick it
)

RUN_DATE = datetime.date.today().isoformat()
WORK = pathlib.Path("week09"); WORK.mkdir(exist_ok=True)

print("panel seed:      ", FILTER_SPEC["seed"])
print("target size:     ", FILTER_SPEC["target_panel_size"])
print("length window:    >=", FILTER_SPEC["min_length"], "to", FILTER_SPEC["max_length"], "(None = no ceiling)")
print("force-included:  ", ", ".join(sorted(ASSIGNED_TARGETS)))
print("run date:        ", RUN_DATE)

---
## Part 1: A panel is a sampling decision, not a data-cleaning chore

### Three identifier systems

| System | Example | What it identifies | How it breaks |
|---|---|---|---|
| UniProt accession | `P0ABQ4` | one protein sequence entry | merged, demerged or deleted between releases; `P00379` is a genuine *secondary* accession for this same entry |
| PDB entry ID | `6XG5` | one deposited experiment, which may contain several different proteins | one protein maps to many entries; one entry maps to many proteins |
| AlphaFold model ID | `AF-P0ABQ4-F1` | one predicted model | derived from the accession at a database release; `-F1` because long sequences are split into overlapping fragments |


### Four ways a join loses rows

1. **The right table genuinely lacks the record.** No AlphaFold model exists for this accession.
2. **The key changed.** Accessions are merged, demerged or deleted between releases, so a link that
   worked last year returns nothing now.
3. **Cardinality.** One protein, five PDB entries. A naive merge does not drop rows here, it
   *multiplies* them, and your panel grows instead of shrinking. 
4. **Coverage policy.** The right table excludes your record on purpose. Varadi et al. (2024) list
   AlphaFold DB's rules: sequences under 16 residues; over 2,700 residues for Swiss-Prot and
   reference proteomes (1,280 elsewhere); non-standard amino acids; anything outside the UniProt
   reference datasets; viral proteins. Those are decisions made by a human!

### Inner or left join

- **AlphaFold: inner join.** No model means no receptor means no docking. The row must go, and it
  goes into the drop log with a reason.
- **PDB: left join.** Most *E. coli* proteins have no experimental structure. That is completely
  normal and is not an indication of a defect. If you do an inner-join here you'll delete most of your panel!


---
## Part 2: The filter specification

The specification is fixed for the whole class. It is in the configuration cell above, and it is
binding — not because it is optimal, but because all our screens have to be comparable. If we filter differently, they are not comparable.

| Rule | Value | Why |
|---|---|---|
| Source | reviewed (Swiss-Prot) entries of `UP000000625` | reviewed means curated; unreviewed TrEMBL would multiply the set by an order of magnitude at much lower annotation quality |
| AlphaFold model | required | no model, no docking receptor |
| Minimum length | 50 aa | below this you are docking into something with no interior |
| **Maximum length** | **none** | see below |
| Fragments | dropped | a fragment record is not a whole protein |
| Non-standard residues | dropped | selenocysteine and friends break the downstream tooling |
| Duplicate sequences | deduplicated | identical sequences are one protein counted twice |
| Panel size | 400, by seeded random subsample | 300–500 by the course design; 150 is the sanctioned fallback |


In [ ]:
#@title The panel logic 
import pandas as pd, numpy as np

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

def classify_entry(row, spec, kept_sequences=None, known_accessions=None):
    """Every reason this entry should be dropped. If empty, it survives.

    row must carry: accession, length, sequence, fragment (bool), has_af_model (bool).
    kept_sequences  a set of sequences already kept, for duplicate detection
    known_accessions if given, an accession outside it is treated as obsolete/demerged
    """
    reasons = []
    if known_accessions is not None and row["accession"] not in known_accessions:
        reasons.append("OBSOLETE_OR_DEMERGED")
    if spec["require_af_model"] and not bool(row.get("has_af_model", False)):
        reasons.append("NO_AF_MODEL")
    n = row.get("length")
    lo, hi = spec["min_length"], spec["max_length"]
    if n is None or (lo is not None and n < lo) or (hi is not None and n > hi):
        reasons.append("LENGTH_OUT_OF_RANGE")
    if spec["drop_fragments"] and bool(row.get("fragment", False)):
        reasons.append("FRAGMENT")
    seq = str(row.get("sequence") or "")
    if spec["drop_nonstandard_residues"] and seq and (set(seq.upper()) - STANDARD_AA):
        reasons.append("NONSTANDARD_RESIDUE")
    if spec["deduplicate_by_sequence"] and kept_sequences is not None and seq in kept_sequences:
        reasons.append("DUPLICATE_SEQUENCE")
    return reasons


def apply_filters(entries, spec, known_accessions=None):
    """Split the input into survivors and a drop log. One drop-log row per dropped accession.

    Deduplication is order-dependent, so the frame is sorted by accession first: the
    pipeline must give the same answer every time it runs, and 'whichever row pandas
    happened to see first' is not the same answer every time.
    """
    df = entries.sort_values("accession").reset_index(drop=True)
    kept_rows, drop_rows, kept_sequences = [], [], set()
    for _, row in df.iterrows():
        reasons = classify_entry(row, spec, kept_sequences, known_accessions)
        if reasons:
            drop_rows.append({
                "accession": row["accession"],
                "entry_name": row.get("entry_name"),
                "gene": row.get("gene"),
                "length": row.get("length"),
                "reason": reasons[0],                 # primary, for counting
                "all_reasons": ";".join(reasons),     # everything, for diagnosis
                "stage": "FILTER",
            })
        else:
            kept_rows.append(row)
            kept_sequences.add(str(row.get("sequence") or "") or f"__no_seq_{row['accession']}")
    kept = pd.DataFrame(kept_rows).reset_index(drop=True)
    drops = pd.DataFrame(drop_rows, columns=["accession", "entry_name", "gene", "length",
                                             "reason", "all_reasons", "stage"])
    return kept, drops


def force_include(entries, kept, drops, targets):
    """Put the assigned targets back, whatever any filter said, and record what it said.

    Returns (kept, drops). A force-included entry carries would_have_dropped_for so the
    override is visible in the shipped file rather than hidden in this function.
    """
    kept = kept.copy()
    kept["force_included"] = kept["accession"].isin(targets)
    kept["would_have_dropped_for"] = ""
    present = set(kept["accession"])
    rescued = []
    for acc in sorted(targets):
        if acc in present:
            continue
        src = entries[entries["accession"] == acc]
        if src.empty:
            raise AssertionError(
                f"ASSIGNED TARGET {acc} IS NOT IN THE SOURCE DOWNLOAD AT ALL. "
                f"Stop. This is a retrieval bug, not a filter decision - do not hand-add the row.")
        why = drops.loc[drops["accession"] == acc, "all_reasons"]
        row = src.iloc[0].to_dict()
        row["force_included"] = True
        row["would_have_dropped_for"] = why.iloc[0] if len(why) else "would_have_passed"
        rescued.append(row)
    if rescued:
        kept = pd.concat([kept, pd.DataFrame(rescued)], ignore_index=True)
    drops = drops[~drops["accession"].isin(targets)].reset_index(drop=True)
    return kept.sort_values("accession").reset_index(drop=True), drops


def seeded_subsample(kept, target_n, seed):
    """Reduce to target_n by seeded random draw. Force-included rows are never sampled.

    Returns (panel, not_sampled_log).
    """
    forced = kept[kept["force_included"]]
    pool = kept[~kept["force_included"]].sort_values("accession").reset_index(drop=True)
    n_draw = max(0, target_n - len(forced))
    if n_draw >= len(pool):
        return kept.sort_values("accession").reset_index(drop=True), \
               pd.DataFrame(columns=["accession", "entry_name", "gene", "length",
                                     "reason", "all_reasons", "stage"])
    rng = np.random.default_rng(seed)
    idx = np.sort(rng.choice(len(pool), size=n_draw, replace=False))
    drawn = pool.iloc[idx]
    left = pool.drop(index=pool.index[idx])
    not_sampled = pd.DataFrame({
        "accession": left["accession"], "entry_name": left.get("entry_name"),
        "gene": left.get("gene"), "length": left.get("length"),
        "reason": "NOT_SAMPLED", "all_reasons": "NOT_SAMPLED", "stage": "SUBSAMPLE",
    })
    panel = pd.concat([forced, drawn], ignore_index=True)
    return panel.sort_values("accession").reset_index(drop=True), not_sampled.reset_index(drop=True)


def reconcile(n_input, panel, drop_log):
    """input = survivors + drops, with no accession on both sides. Raises if not."""
    both = set(panel["accession"]) & set(drop_log["accession"])
    assert not both, f"accessions in BOTH panel and drop log: {sorted(both)[:10]}"
    assert drop_log["accession"].is_unique, "an accession is logged as dropped more than once"
    assert set(drop_log["reason"]) <= set(DROP_REASONS), \
        f"reason codes outside the closed vocabulary: {set(drop_log['reason']) - set(DROP_REASONS)}"
    total = len(panel) + len(drop_log)
    assert total == n_input, f"{n_input} in, {len(panel)} kept + {len(drop_log)} dropped = {total}"
    return True

print("panel logic defined: classify_entry, apply_filters, force_include, seeded_subsample, reconcile")

In [ ]:
#@title Self-test on an INVENTED frame 
# These twelve rows are made up. The sequences are nonsense and the lengths are chosen to
# exercise each branch. Nothing here is a claim about any real protein. Run this cell first:
#  if it fails, your pipeline is broken.
import pandas as pd, numpy as np

def _fake(acc, length, af=True, frag=False, seq=None, gene="x"):
    return {"accession": acc, "entry_name": acc + "_TEST", "gene": gene, "length": length,
            "sequence": seq if seq is not None else "A" * length,
            "fragment": frag, "has_af_model": af,
            "protein_name": "invented test entry", "pdb_ids": "", "n_pdb": 0}

synthetic = pd.DataFrame([
    _fake("TEST01", 120),                             # survives
    _fake("TEST02", 300),                             # survives
    _fake("TEST03", 45),                              # too short
    _fake("TEST04", 200, af=False),                   # no AlphaFold model
    _fake("TEST05", 180, frag=True),                  # fragment
    _fake("TEST06", 150, seq="U" + "A" * 149),        # selenocysteine
    _fake("TEST07", 120),                             # duplicate of TEST01 by sequence
    _fake("TEST08", 2000),                            # long, and MUST survive (no ceiling)
    _fake("P0ABQ4", 159, gene="folA"),                # target, passes anyway
    _fake("P0AES4", 875, gene="gyrA"),                # target, 875 aa - dies under any ceiling
    _fake("P0AES6", 804, af=False, gene="gyrB"),      # target with NO MODEL - force-include must win
    _fake("P0AC13", 282, frag=True, gene="folP"),     # target marked fragment - force-include wins
])
# The fifth target is deliberately absent, to ensure the retrieval-bug assertion fires.

kept_s, drops_s = apply_filters(synthetic, FILTER_SPEC)
print("after filters:", len(kept_s), "kept,", len(drops_s), "dropped")
print(drops_s[["accession", "reason", "all_reasons"]].to_string(index=False))

assert "TEST08" in set(kept_s["accession"]), "a 2000 aa entry was dropped - someone added a ceiling"
assert set(drops_s.loc[drops_s["reason"] == "LENGTH_OUT_OF_RANGE", "accession"]) == {"TEST03"}
assert "DUPLICATE_SEQUENCE" in set(drops_s["all_reasons"].str.cat(sep=";").split(";"))

try:
    force_include(synthetic, kept_s, drops_s, set(ASSIGNED_TARGETS))
except AssertionError as e:
    print()
    print("EXPECTED failure, because P0AEK4 is missing from this invented frame:")
    print(" ", str(e)[:140], "...")

kept_s, drops_s = force_include(synthetic, kept_s, drops_s, set(ASSIGNED_TARGETS) - {"P0AEK4"})
rescued = kept_s[kept_s["force_included"]][["accession", "gene", "length", "would_have_dropped_for"]]
print()
print("force-included rows, with what the filter would have done to them:")
print(rescued.to_string(index=False))
assert {"P0AES6", "P0AC13"} <= set(kept_s["accession"]), "force-include failed to override a filter"
assert not (set(drops_s["accession"]) & set(ASSIGNED_TARGETS)), "a target is still in the drop log"

panel_s, not_sampled_s = seeded_subsample(kept_s, target_n=6, seed=FILTER_SPEC["seed"])
log_s = pd.concat([drops_s, not_sampled_s], ignore_index=True)
assert set(panel_s.loc[panel_s["force_included"], "accession"]) == \
       (set(ASSIGNED_TARGETS) - {"P0AEK4"}), "subsampling removed a force-included target"
reconcile(len(synthetic), panel_s, log_s)
print()
print(f"reconciled: {len(synthetic)} in = {len(panel_s)} panel + {len(log_s)} logged drops")

again, _ = seeded_subsample(kept_s, target_n=6, seed=FILTER_SPEC["seed"])
assert list(again["accession"]) == list(panel_s["accession"]), "the seed is not doing its job"
print("same seed, same panel - the subsample is reproducible")
print()
print("SELF-TEST PASSED. The rows above are invented and are not results.")

---
## Part 3: Getting the proteome 


In [ ]:
#@title NETWORK - stream the reviewed UP000000625 proteome (cache first)
import pathlib, requests, io, pandas as pd

RAW = WORK / "week09_uniprot_raw.tsv"     # the cache. Delete it to force a re-download.
FIELDS = ("accession,id,protein_name,gene_primary,length,fragment,sequence,"
          "xref_alphafolddb,xref_pdb")
QUERY = f"(proteome:{FILTER_SPEC['proteome']}) AND (reviewed:true)"
STREAM = "https://rest.uniprot.org/uniprotkb/stream"

uniprot_release = "UNKNOWN"
if RAW.exists() and RAW.stat().st_size > 0:
    print(f"cache hit: {RAW} ({RAW.stat().st_size:,} bytes) - delete it to re-download")
else:
    try:
        t0 = time.time()
        r = requests.get(STREAM, params={"query": QUERY, "format": "tsv", "fields": FIELDS},
                         timeout=300)
        r.raise_for_status()
        RAW.write_text(r.text)
        uniprot_release = r.headers.get("X-UniProt-Release", "UNKNOWN")
        print(f"downloaded {len(r.text):,} bytes in {time.time() - t0:.1f} s")
        print("UniProt release header:", uniprot_release, "  (record this number)")
    except Exception as e:
        print("DOWNLOAD FAILED:", type(e).__name__, str(e)[:300])
        print()
        print("Fallback, in order:")
        print("  1. Use the cached TSV your instructor put in the shared Colab folder.")
        print("  2. Open the URL below in a browser, save the result, upload it as", RAW)
        print(f"     {STREAM}?query={QUERY}&format=tsv&fields={FIELDS}")
        print("  3. Everything after this point still runs on the synthetic frame, and the")
        print("     logic you are graded on is the same logic.")

raw = None
if RAW.exists() and RAW.stat().st_size > 0:
    raw = pd.read_csv(RAW, sep="\t")
    print()
    print(f"rows returned: {len(raw):,}")
    print("columns:", list(raw.columns))
    print()
    print("CHECK THIS NUMBER. If it is exactly 500, you hit the paginated endpoint and your")
    print("panel would have been built from a truncated download with no error message.")
    if len(raw) == 500:
        print(">>> 500 rows exactly. Treat this as a truncation until you have proved otherwise. <<<")

---
## Part 4: The two joins



In [ ]:
#@title Normalise and join - falls back to the synthetic frame if the download failed
import pandas as pd, numpy as np

def normalise_uniprot(df):
    """UniProt TSV -> the column names the panel logic expects. One row per accession."""
    out = pd.DataFrame({
        "accession":   df["Entry"].astype(str).str.strip(),
        "entry_name":  df.get("Entry Name"),
        "protein_name": df.get("Protein names"),
        "gene":        df.get("Gene Names (primary)"),
        "length":      pd.to_numeric(df.get("Length"), errors="coerce").astype("Int64"),
        "sequence":    df.get("Sequence").astype(str).str.strip().str.upper(),
        # UniProt writes "fragment" / "fragments" in this column and leaves it blank otherwise
        "fragment":    df.get("Fragment").fillna("").astype(str).str.strip().ne(""),
    })
    af = df.get("AlphaFoldDB").fillna("").astype(str)
    out["af_xref"] = af.str.replace(";", "", regex=False).str.strip()
    out["has_af_model"] = out["af_xref"].ne("")
    # derive the model identifier from the accession rather than trusting the xref string
    out["af_model_id"] = np.where(out["has_af_model"], "AF-" + out["accession"] + "-F1", "")

    # PDB: AGGREGATE, then join. Never merge a one-to-many relation raw.
    pdb = df.get("PDB").fillna("").astype(str)
    ids = pdb.apply(lambda s: sorted({p.strip() for p in s.split(";") if p.strip()}))
    out["pdb_ids"] = ids.apply(";".join)
    out["n_pdb"] = ids.apply(len)
    return out.drop_duplicates(subset="accession").reset_index(drop=True)

USED_REAL_DATA = False
try:
    entries = normalise_uniprot(raw)
    USED_REAL_DATA = True
except Exception as e:
    print("could not normalise the download -", type(e).__name__, str(e)[:200])
    print("FALLING BACK TO THE INVENTED FRAME. Everything below is logic, not results.")
    entries = synthetic.copy()
    entries["af_model_id"] = np.where(entries["has_af_model"],
                                      "AF-" + entries["accession"] + "-F1", "")
    entries["protein_name"] = entries.get("protein_name", "invented test entry")

print("DATA SOURCE:", "real UniProt download" if USED_REAL_DATA else "INVENTED SYNTHETIC FRAME")
print(f"entries: {len(entries):,}")
print()
print("--- join 1: AlphaFold DB, INNER (no model, no receptor) ---")
n_af = int(entries["has_af_model"].sum())
print(f"  with a model:    {n_af:,}")
print(f"  without a model: {len(entries) - n_af:,}   <- these will be dropped, with a reason")
print()
print("--- join 2: PDB, LEFT and AGGREGATED (no structure is normal) ---")
print(f"  with >=1 PDB entry: {int((entries['n_pdb'] > 0).sum()):,}")
print(f"  with none:          {int((entries['n_pdb'] == 0).sum()):,}   <- kept. Not a defect.")
print(f"  total PDB entries referenced: {int(entries['n_pdb'].sum()):,}")
print()
print("Note the last number. A naive one-to-many merge on PDB would have produced roughly")
print("that many rows instead of one per protein, and the panel would have SILENTLY GROWN.")
print()
print("--- the five assigned targets, before any filtering ---")
chk = entries[entries["accession"].isin(ASSIGNED_TARGETS)][
    ["accession", "gene", "length", "has_af_model", "n_pdb"]]
print(chk.to_string(index=False) if len(chk) else "  NONE FOUND - check the download before going on")

---
## Part 4b: Build the panel

Now put it together.

1. filter → 2. force-include the five targets → 3. seeded subsample → 4. reconcile → 5. write.

Force-inclusion happens **after** filtering, not before, so that the drop log records what the filter
*would* have done to each target. 

Write two files, both registered course artifacts:

- **`week09_panel.csv`** — one row per protein, consumed by Week 12.
- **`week09_drop_log.csv`** — one row per dropped accession, **graded**.

In [ ]:
# ---- YOUR CODE HERE ----
# Expected: week09_panel.csv and week09_drop_log.csv written; the five assigned targets asserted present; input = survivors + drops asserted; a printed count of drops by reason code; the seed and the query date recorded in the file itself
# Example: panel: 400 rows, of which 5 force-included; drop log: (input - 400) rows across the reason codes, with NOT_SAMPLED the largest single category
# ---- END ----


---
## Part 5: THE FREEZE

**The panel is now frozen. Nothing in `week09_panel.csv` changes for the rest of the term.**



---
## Part 6: Pockets for the whole panel



In [ ]:
#@title NETWORK - fetch AlphaFold models for the panel, chunked and cached
import pathlib, requests, time, pandas as pd

MODELS = WORK / "models"; MODELS.mkdir(exist_ok=True)
AF_VERSIONS = ["v6", "v5"]        
CHUNK_SIZE = 25
MAX_MINUTES_PER_RUN = 4.0         # so this cell obeys the five-minute rule; re-run it to continue

try:
    panel_ids = list(pd.read_csv("week09_panel.csv")["accession"])
except Exception as e:
    print("week09_panel.csv not readable -", type(e).__name__, str(e)[:150])
    panel_ids = []

def model_path(acc):
    return MODELS / f"AF-{acc}-F1.pdb"

def fetch_model(acc, timeout=60):
    """Return (path, note). Tries each version suffix. Cached files are never re-fetched."""
    p = model_path(acc)
    if p.exists() and p.stat().st_size > 0:
        return p, "cached"
    for v in AF_VERSIONS:
        url = f"https://alphafold.ebi.ac.uk/files/AF-{acc}-F1-model_{v}.pdb"
        try:
            r = requests.get(url, timeout=timeout)
            if r.status_code == 200 and r.content:
                p.write_bytes(r.content)
                return p, v
        except Exception as e:
            return None, f"{type(e).__name__}"
    return None, "not_found"

todo = [a for a in panel_ids if not (model_path(a).exists() and model_path(a).stat().st_size > 0)]
print(f"panel: {len(panel_ids)}   already on disk: {len(panel_ids) - len(todo)}   to fetch: {len(todo)}")

t0, fetched, failed = time.time(), 0, []
for i, acc in enumerate(todo):
    if (time.time() - t0) / 60 > MAX_MINUTES_PER_RUN:
        print(f"\ntime-boxed after {i} of {len(todo)}. RE-RUN THIS CELL to continue - it resumes.")
        break
    p, note = fetch_model(acc)
    if p is None:
        failed.append((acc, note))
    else:
        fetched += 1
    if i and i % CHUNK_SIZE == 0:
        rate = (time.time() - t0) / i
        print(f"  {i}/{len(todo)}  {rate:.2f} s/model  "
              f"projected remaining {(len(todo) - i) * rate / 60:.1f} min")
    time.sleep(0.1)      # be a good citizen; this is somebody's server

on_disk = sum(1 for a in panel_ids if model_path(a).exists() and model_path(a).stat().st_size > 0)
print()
print(f"models on disk: {on_disk}/{len(panel_ids)}   this run fetched {fetched} in "
      f"{(time.time() - t0) / 60:.1f} min")
if failed:
    print(f"failed: {len(failed)} - first few: {failed[:5]}")
    print("An accession with a UniProt AlphaFold cross-reference but no downloadable model is a")
    print("real finding, not a nuisance. Note it: the cross-reference and the model live in")
    print("different databases on different release schedules.")
if on_disk < len(panel_ids):
    print()
    print("Alternative if per-accession fetching is too slow: take the per-organism bulk archive")
    print("from ftp.ebi.ac.uk/pub/databases/alphafold/ and extract only the panel accessions.")
    print("[VERIFY the current E. coli archive filename - it carries a version suffix that moves.]")

In [ ]:
#@title Build fpocket, then run it across the panel
import subprocess, shutil, pathlib, time, numpy as np, pandas as pd

CHECKPOINTS = WORK / "pocket_chunks"; CHECKPOINTS.mkdir(exist_ok=True)
def _colab_build_fpocket():
    """Build fpocket from source. Only called on Colab, where the VM is disposable."""
    print("  building fpocket from source (about a minute, once per runtime)...")
    subprocess.run("apt-get -qq install -y build-essential libnetcdf-dev git",
                   shell=True, check=True, capture_output=True, timeout=900)
    subprocess.run("git clone -q --depth 1 https://github.com/Discngine/fpocket.git /tmp/fpocket",
                   shell=True, check=True, capture_output=True, timeout=900)
    subprocess.run("make -s -C /tmp/fpocket && make -s -C /tmp/fpocket install",
                   shell=True, check=True, capture_output=True, timeout=900)

check_tools(["fpocket"], colab_installer=_colab_build_fpocket)
FPOCKET = shutil.which("fpocket")

if FPOCKET is None:
    print("Use your instructor's cached pocket chunks in", CHECKPOINTS)
    print("This week's panel-wide run is the one step that needs fpocket locally;")
    print("on Windows without WSL, run this notebook on Colab instead.")

def parse_pqr_centres(pqr):
    """Pocket centre = mean alpha-sphere centre. Same parser as Week 8, field-split not fixed-column."""
    groups = {}
    for line in pathlib.Path(pqr).read_text().splitlines():
        if not line.startswith(("ATOM", "HETATM")):
            continue
        f = line.split()
        try:
            pid = int(f[5]); xyz = (float(f[6]), float(f[7]), float(f[8]))
        except (ValueError, IndexError):
            continue
        groups.setdefault(pid, []).append(xyz)
    return {k: tuple(np.mean(v, axis=0)) for k, v in groups.items()}

def parse_info(info):
    """Per-pocket descriptors from *_info.txt -> {pocket_id: {label: float}}."""
    out, cur = {}, None
    for line in pathlib.Path(info).read_text().splitlines():
        s = line.strip()
        if s.lower().startswith("pocket") and ":" in s:
            try: cur = int(s.split()[1])
            except (ValueError, IndexError): cur = None
            if cur is not None: out[cur] = {}
        elif cur is not None and ":" in s:
            k, _, v = s.partition(":")
            try: out[cur][k.strip()] = float(v.strip())
            except ValueError: pass
    return out

def parse_lining(pocket_atm):
    """Residues lining one pocket, from pockets/pocketN_atm.pdb -> 'A:ILE5;A:ALA6;...'"""
    seen = []
    for line in pathlib.Path(pocket_atm).read_text().splitlines():
        if not line.startswith("ATOM"):
            continue
        try:
            resn, ch, num = line[17:20].strip(), line[21].strip() or "A", int(line[22:26])
        except ValueError:
            continue
        key = f"{ch}:{resn}{num}"
        if key not in seen:
            seen.append(key)
    return ";".join(seen)

def run_one(acc):
    """fpocket on one AlphaFold model -> list of row dicts. Empty list means no pockets."""
    src = model_path(acc)
    if not (src.exists() and src.stat().st_size > 0):
        return []
    out = src.parent / f"{src.stem}_out"
    if not out.exists():
        subprocess.run([FPOCKET, "-f", str(src)], check=True, capture_output=True, timeout=600)
    centres, info = parse_pqr_centres(out / f"{src.stem}_pockets.pqr"), parse_info(out / f"{src.stem}_info.txt")
    rows = []
    for pid, xyz in sorted(centres.items()):
        d = info.get(pid, {})
        atm = out / "pockets" / f"pocket{pid}_atm.pdb"
        rows.append({
            "accession": acc, "pocket_rank": pid,
            "score": d.get("Score"), "volume": d.get("Volume"),
            "n_alpha_spheres": d.get("Number of Alpha Spheres"),
            "center_x": round(float(xyz[0]), 3), "center_y": round(float(xyz[1]), 3),
            "center_z": round(float(xyz[2]), 3),
            "lining_residues": parse_lining(atm) if atm.exists() else "",
        })
    return rows

chunks = [panel_ids[i:i + CHUNK_SIZE] for i in range(0, len(panel_ids), CHUNK_SIZE)]
print(f"{len(panel_ids)} proteins in {len(chunks)} chunks of {CHUNK_SIZE}")
t0, done_now, per_protein = time.time(), 0, []

for ci, chunk in enumerate(chunks):
    ck = CHECKPOINTS / f"chunk_{ci:03d}.csv"
    if ck.exists():
        continue                                  # idempotent: finished chunks are never redone
    if FPOCKET is None:
        print("no fpocket binary and no checkpoint for chunk", ci, "- stopping")
        break
    if (time.time() - t0) / 60 > MAX_MINUTES_PER_RUN:
        print(f"time-boxed. {ci} of {len(chunks)} chunks done. RE-RUN THIS CELL to continue.")
        break
    rows = []
    for acc in chunk:
        s = time.time()
        try:
            rows.extend(run_one(acc))
        except Exception as e:
            print(f"  {acc}: fpocket failed - {type(e).__name__}: {str(e)[:120]}")
        per_protein.append(time.time() - s)
    pd.DataFrame(rows, columns=["accession", "pocket_rank", "score", "volume", "n_alpha_spheres",
                                "center_x", "center_y", "center_z", "lining_residues"]
                 ).to_csv(ck, index=False)
    done_now += 1
    med = float(np.median(per_protein)) if per_protein else float("nan")
    print(f"  chunk {ci:3d}/{len(chunks)}  {len(rows):4d} pockets  "
          f"median {med:.2f} s/protein  projected total "
          f"{med * len(panel_ids) / 60:.1f} min")

finished = sorted(CHECKPOINTS.glob("chunk_*.csv"))
print()
print(f"chunks complete: {len(finished)}/{len(chunks)}   this run: {done_now} in "
      f"{(time.time() - t0) / 60:.1f} min")
if per_protein:
    a = np.array(per_protein)
    print(f"RUNTIME: median {np.median(a):.2f} s/protein, mean {a.mean():.2f}, max {a.max():.2f}")
    print(f"         projected wall time for {len(panel_ids)} proteins: "
          f"{np.median(a) * len(panel_ids) / 60:.1f} min")
    print("Report these numbers. 'It took a while' is not a runtime report.")
if len(finished) < len(chunks):
    print("NOT FINISHED. Re-run this cell. It resumes from the checkpoints and skips what is done.")

In [ ]:
# ---- YOUR CODE HERE ----
# Expected: week09_panel_pockets.csv assembled from the checkpoint chunks with exactly the columns accession, pocket_rank, score, volume, n_alpha_spheres, center_x, center_y, center_z, lining_residues; plus a coverage check against week09_panel.csv, a distribution of pockets per protein, and a stated runtime
# Example: week09_panel_pockets.csv: several thousand rows over the panel; every panel accession either present or listed as producing no pockets, with the count of each stated
# ---- END ----


In [ ]:
# ---- YOUR CODE HERE ----
# Expected: hashes proving two runs agree; week09_manifest.json with release, query, seed, filter spec, tool versions and per-file hashes; week09_data_dictionary.md with one entry per column of both files; and a reusable check_panel() assertion suite that Week 12 can import
# Example: check_panel('week09_panel.csv') returns a dict of passed checks and raises on the first failure; running the pipeline twice gives identical SHA-256 for week09_panel.csv
# ---- END ----


In [ ]:
# ---- YOUR CODE HERE ----
# Expected: a table of ten diagnosed drops with protein class, proximate reason and underlying property; a 300-400 word account of the systematic bias naming at least three under-represented classes and the direction of each effect; and one sentence naming a finding the Week 12 screen cannot produce
# Example: of ten drops, four are membrane-associated and three are short/fragment records, so the panel is depleted of exactly the proteins whose binding sites are hardest to define
# ---- END ----
